In [13]:
from eventregistry import EventRegistry, QueryArticlesIter, QueryItems
import os
import json

In [14]:
def run_newsapi_ai_scraper(keywords,starting_date,ending_date):
    
    API_KEY = "5ff63ede-ac37-47f4-b8c3-ab2ebaece1ab"

    er = EventRegistry(apiKey = API_KEY, allowUseOfArchive = False)

    q = QueryArticlesIter(
        keywords = QueryItems.OR(keywords),
        # lang=QueryItems.OR(["eng",]),
        sourceLocationUri="http://en.wikipedia.org/wiki/Malaysia",
        dateStart=starting_date,
        dateEnd=ending_date,
        keywordsLoc = "body,title",
        keywordSearchMode = "exact",
        isDuplicateFilter="skipDuplicates",
        hasDuplicateFilter="skipHasDuplicates",
        dataType= ["news", "pr"],
        )

    all_articles = []
    try: 
        # we limit here the results to 100. If you want more, remove or increase maxItems
        for article in q.execQuery(er, sortBy="date", sortByAsc=False):
            all_articles.append(article)

        print(f"\nSuccessfully retrieved {len(all_articles)} articles.")

        # Process sentiment for each news item
        print("\nProcessing sentiment analysis:")
        for article in all_articles:
            if 'sentiment' in article:
                article['sentiment_score']=article.pop('sentiment') # .pop means take this key out of the dict and return its value
            sentiment_score = article.get("sentiment_score",0)
            if sentiment_score is None:
                sentiment_score = 0
            # Classify sentiment based on score
            if sentiment_score > 0.2:
                article['sentiment'] = 'Positive'
                print(f"[Positive] {article.get('title','No Title')}")
            elif sentiment_score < -0.2:
                article['sentiment'] = 'Negative'
                print(f"[Negative] {article.get('title','No Title')}")
            else:
                article['sentiment'] = 'Neutral'
                print(f"[Neutral] {article.get('title','No Title')}")

            # standardize field name for convert_news_to_structured function
            try:
                article["id"] = article.pop("uri")
                article["published_date"] = article.pop("dateTimePub")
                article["text"] = article.pop("body")
                article["language"] = article.pop("lang")
            except KeyError as e:
                print(f"Warning: Missing field {e} in article: {article.get('title', 'No title')}")
                continue  # Skip this article
            
        # Save to JSON
        output_dir = "logs"
        # os.makedirs(output_dir, exist_ok=True)
        filename = "newsapi_ai_results.json"
        filepath = os.path.join(output_dir, filename)

        output = {
            "total_available": len(all_articles),
            # "max_results": max_results,
            # "date_range": {"start": starting_date, "end": ending_date},
            # "keywords": keywords,
            "news": all_articles
        }

        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=4, ensure_ascii=False)

        print(f"All articles saved to '{filepath}'")
        return output
        
    except Exception as e:
        print(f"An error occurred: {e}")
        return False
    except ValueError:
        print("Could not decode the JSON response.")
        return False
    except IOError as e:
        print(f"An error occurred while writing to the file: {e}")
        return False

In [15]:
keywords = ["Akta Syarikat 1965","Akta Syarikat 2016","Akta Syarikat (Pindaan) 2024","SSM4U","Malaysian Business Reporting System"]
            # Ketua Pegawai Eksekutif SSM,Akta (Larangan) Kumpulan Wang Kutu 1971,Akta Pendaftaran Perniagaan 1956,Akta Perkongsian Liabiliti Terhad 2012,Akta Skim Kepentingan 2016,Akta Syarikat Amanah 1949,Peraturan-Peraturan Skim Kepentingan 2017,Peraturan-Peraturan Syarikat 2017,Kaedah-Kaedah Pendaftaran Perniagaan 1957,Peraturan-Peraturan Suruhanjaya Syarikat Malaysia (Perlesenan Setiausaha 2017),Peraturan-Peraturan Perkongsian Liabiliti Terhad 2012,Electronic Beneficial Ownership System,EzBiz Online,MyCoID,MyLLP,Syarikat Berhad Menurut Jaminan]
starting_date = "2025-09-18"
ending_date = "2025-10-31"

In [12]:
json_output = run_newsapi_ai_scraper(
    keywords,
    starting_date,
    ending_date,
    # max_results=100
)

Error while obtaining a list of articles: Too many keywords specified in the query. You've specified 25, while the allowed number for your subscription type is 15



Successfully retrieved 0 articles.

Processing sentiment analysis:
An error occurred: name 'json' is not defined


In [6]:
json_output

{'total_fetched': 3,
 'max_results': 3,
 'date_range': {'start': '2025-09-18', 'end': '2025-10-31'},
 'keywords': ['SSM', '大马公司委员会'],
 'articles': [{'uri': '2025-10-878517280',
   'lang': 'eng',
   'isDuplicate': False,
   'date': '2025-10-31',
   'time': '05:28:54',
   'dateTime': '2025-10-31T05:28:54Z',
   'dateTimePub': '2025-10-31T00:00:00Z',
   'dataType': 'news',
   'sim': 0,
   'url': 'https://www.dailyexpress.com.my/news/269450/round-the-clock-worker-protection-soon-perkeso/',
   'title': 'Round-the-clock worker protection soon: Perkeso',
   'body': 'PUTRAJAYA: About 10 million formal sector workers nationwide will soon enjoy comprehensive social security coverage through the new Non-Work-Related Accident Scheme, or Lindung 24/7, under the LINDUNG Pekerja programme by the Social Security Organisation (Perkeso).\n\nIn a statement, the Ministry of Human Resources (KESUMA) said the scheme provides continuous protection - 24 hours a day, seven days a week, extending coverage beyond

In [8]:
json_output.get('articles', '').get('source', '').get('title','') # Display first 2 articles

AttributeError: 'list' object has no attribute 'get'

In [22]:
len(json_output)

3

In [ ]:
import urllib.parse
text_query = "earthquake OR tsunami"
text = 


In [11]:
print(text_param)

earthquake%20OR%20tsunami
